In [ ]:
from pathlib import Path
import ast

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Carpeta principal del proyecto
BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

# Carpetas principales
DATA_DIR = BASE_DIR / "data"
REY_DIR = DATA_DIR / "REY_DATASET"

REY_SCAN_DIR = REY_DIR / "REY_scan_anonim"
REY_ROI_ROT0_DIR = REY_DIR / "REY_roi_rot0"
REY_ROI_MANUAL_DIR = REY_DIR / "REY_roi_manualSelection1"

TRAZA_CSV = REY_DIR / "traza_REY.csv"
ANOTACIONES_CSV = REY_DIR / "anotaciones.csv"

# Salidas
OUTPUT_DIR = BASE_DIR / "outputs"
FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("REY_DIR existe:", REY_DIR.exists())
print("REY_SCAN_DIR existe:", REY_SCAN_DIR.exists())
print("REY_ROI_ROT0_DIR existe:", REY_ROI_ROT0_DIR.exists())
print("REY_ROI_MANUAL_DIR existe:", REY_ROI_MANUAL_DIR.exists())
print("TRAZA_CSV existe:", TRAZA_CSV.exists())

In [ ]:
import re

filas = []

with open(TRAZA_CSV, "r", encoding="utf-8", errors="ignore") as f:
    lineas = f.readlines()

for linea in lineas:
    linea = linea.strip()
    
    if linea == "":
        continue
    
    # Extraer id
    id_match = re.search(r"id:(\d+)", linea)
    
    # Extraer fname
    fname_match = re.search(r"fname:([^;]+)", linea)
    
    # Extraer forig
    forig_match = re.search(r"forig:([^;]+)", linea)
    
    # Extraer BB: cualquier cosa entre [ ]
    bb_match = re.search(r"BB:\[([^\]]+)\]", linea)
    
    # Extraer rot
    rot_match = re.search(r"rot:([-]?\d+)", linea)
    
    if id_match and fname_match and forig_match and bb_match and rot_match:
        id_val = int(id_match.group(1))
        fname_val = fname_match.group(1).strip()
        forig_val = forig_match.group(1).strip()
        rot_val = int(rot_match.group(1))
        
        # La BB puede venir separada por comas, espacios o tabuladores
        bb_texto = bb_match.group(1)
        bb_nums = re.findall(r"-?\d+", bb_texto)
        bb_nums = [int(x) for x in bb_nums]
        
        if len(bb_nums) == 4:
            x = bb_nums[0]
            y = bb_nums[1]
            h = bb_nums[2]
            w = bb_nums[3]
            
            filas.append({
                "id": id_val,
                "fname": fname_val,
                "forig": forig_val,
                "x": x,
                "y": y,
                "h": h,
                "w": w,
                "rot": rot_val
            })

df = pd.DataFrame(filas)

print("Número de líneas originales:", len(lineas))
print("Número de filas válidas:", len(df))
print("Columnas:", df.columns.tolist())

df.head()

In [ ]:
print("Primeras filas limpias:")
display(df.head())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores de rotación disponibles:")
print(df["rot"].value_counts(dropna=False).sort_index())

print("\nResumen de bounding boxes:")
display(df[["x", "y", "h", "w"]].describe())

In [ ]:
def contar_imagenes(carpeta):
    extensiones = ["*.jpg", "*.jpeg", "*.png", "*.bmp"]
    ficheros = []
    
    for ext in extensiones:
        ficheros.extend(list(carpeta.glob(ext)))
    
    return ficheros

scan_files = contar_imagenes(REY_SCAN_DIR)
roi_rot0_files = contar_imagenes(REY_ROI_ROT0_DIR)
roi_manual_files = contar_imagenes(REY_ROI_MANUAL_DIR)

print("Imágenes originales:", len(scan_files))
print("ROIs rot0:", len(roi_rot0_files))
print("ROIs manualSelection1:", len(roi_manual_files))

print("\nEjemplo original:", scan_files[0].name if len(scan_files) > 0 else "No hay")
print("Ejemplo ROI rot0:", roi_rot0_files[0].name if len(roi_rot0_files) > 0 else "No hay")
print("Ejemplo ROI manual:", roi_manual_files[0].name if len(roi_manual_files) > 0 else "No hay")

In [ ]:
def leer_imagen(path, color=True):
    if color:
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    else:
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    
    if img is None:
        print("No se pudo leer:", path)
    
    return img


def parsear_bb(bb_texto):
    """
    Convierte un texto tipo '[187, 221, 200, 182]'
    en x, y, h, w.
    """
    valores = ast.literal_eval(bb_texto)
    
    x = int(valores[0])
    y = int(valores[1])
    h = int(valores[2])
    w = int(valores[3])
    
    return x, y, h, w

In [ ]:
fila = df.iloc[0]

fname_roi = fila["fname"]
fname_original = fila["forig"]
rot = fila["rot"]

print("ROI:", fname_roi)
print("Original:", fname_original)
print("x, y, h, w:", fila["x"], fila["y"], fila["h"], fila["w"])
print("Rotación:", rot)

img_path = REY_SCAN_DIR / fname_original
img = leer_imagen(img_path, color=True)

x = int(fila["x"])
y = int(fila["y"])
h = int(fila["h"])
w = int(fila["w"])

img_dib = img.copy()
cv2.rectangle(img_dib, (x, y), (x + w, y + h), (0, 0, 255), 3)

img_rgb = cv2.cvtColor(img_dib, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(7, 7))
plt.imshow(img_rgb)
plt.title("Imagen original con BB del CSV")
plt.axis("off")
plt.show()

In [ ]:
roi_path = REY_ROI_ROT0_DIR / fname_roi

roi_img = leer_imagen(roi_path, color=True)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.title("Original")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(cv2.cvtColor(roi_img, cv2.COLOR_BGR2RGB))
plt.title("ROI rot0")
plt.axis("off")

plt.show()

In [ ]:
n_muestras = 6
muestras = df.sample(n=min(n_muestras, len(df)), random_state=7)

plt.figure(figsize=(12, 8))

for i, (_, fila) in enumerate(muestras.iterrows()):
    img_path = REY_SCAN_DIR / fila["forig"]
    img = leer_imagen(img_path, color=True)
    
    if img is None:
        continue
    
    x = int(fila["x"])
    y = int(fila["y"])
    h = int(fila["h"])
    w = int(fila["w"])
    
    img_dib = img.copy()
    cv2.rectangle(img_dib, (x, y), (x + w, y + h), (0, 0, 255), 3)
    
    img_rgb = cv2.cvtColor(img_dib, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 3, i + 1)
    plt.imshow(img_rgb)
    plt.title(f"rot={fila['rot']}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
fig_path = FIGURES_DIR / "rey_ejemplos_bb.png"

plt.figure(figsize=(12, 8))

for i, (_, fila) in enumerate(muestras.iterrows()):
    img_path = REY_SCAN_DIR / fila["forig"]
    img = leer_imagen(img_path, color=True)
    
    if img is None:
        continue
    
    x = int(fila["x"])
    y = int(fila["y"])
    h = int(fila["h"])
    w = int(fila["w"])
    
    img_dib = img.copy()
    cv2.rectangle(img_dib, (x, y), (x + w, y + h), (0, 0, 255), 3)
    
    img_rgb = cv2.cvtColor(img_dib, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 3, i + 1)
    plt.imshow(img_rgb)
    plt.title(f"rot={fila['rot']}")
    plt.axis("off")

plt.tight_layout()
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada en:", fig_path)

In [ ]:
# Comprobamos si los archivos mencionados en el CSV existen realmente

existe_original = []
existe_roi_rot0 = []

for i, fila in df.iterrows():
    path_original = REY_SCAN_DIR / fila["forig"]
    path_roi = REY_ROI_ROT0_DIR / fila["fname"]
    
    existe_original.append(path_original.exists())
    existe_roi_rot0.append(path_roi.exists())

df["existe_original"] = existe_original
df["existe_roi_rot0"] = existe_roi_rot0

print("Filas totales:", len(df))
print("Originales existentes:", df["existe_original"].sum())
print("ROIs rot0 existentes:", df["existe_roi_rot0"].sum())

df_problematicas = df[(df["existe_original"] == False) | (df["existe_roi_rot0"] == False)]

print("Filas problemáticas:", len(df_problematicas))
display(df_problematicas)

In [ ]:
# Nos quedamos solo con las filas que tienen imagen original y ROI rot0

df_clean = df[(df["existe_original"] == True) & (df["existe_roi_rot0"] == True)].copy()

print("Filas limpias finales:", len(df_clean))

print("\nDistribución de rotación tras limpiar:")
print(df_clean["rot"].value_counts().sort_index())

display(df_clean.head())

In [ ]:
csv_limpio = TABLES_DIR / "rey_traza_limpia.csv"

df_clean.to_csv(csv_limpio, index=False)

print("CSV limpio guardado en:")
print(csv_limpio)